# PS S6E4 - CatBoost no pairwise, keep original + bias

目的:
- exp_20260407_018_cat_pairwise_te_bias_from_shared_min から pairwise 関連だけを除去
- original 系と bias tuning は維持
- 018 の強さにおける pairwise の寄与を切り分ける

仮説:
- 021 が 018 から大きく落ちるなら pairwise は主役級
- 021 があまり落ちないなら original + bias の寄与が大きい
- 021 と 020 の比較で pairwise vs original の重さを判断する

## Imports / config

In [1]:
import os
import json
import warnings
import random
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from catboost import CatBoostClassifier, Pool
import torch

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)


class CFG:
    EXP_ID = "exp_20260407_021_cat_no_pairwise_bias"
    NOTEBOOK_TAG = "cat_no_pairwise_bias"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    ORIGINAL_PATH = "/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv"
    OUTPUT_DIR = Path(f"./experiments/{EXP_ID}/artifacts")

    ID_COL = "id"
    TARGET_COL = "Irrigation_Need"
    SEED = 42
    N_FOLDS = 5
    INNER_FOLDS_TE = 3  # 021 では未使用だが記録として残す

    USE_PHYSICAL_FEATURES = True
    USE_ORIG_PRIORS = True
    USE_ORIGINAL_WEIGHTED_MERGE = True
    ORIG_ROW_WEIGHT = 0.35
    USE_BIAS_TUNING = True

    # 021 の核心
    USE_PAIRWISE = False
    USE_PAIRWISE_TE = False

    N_BINS_ORIG_PRIOR = 10

    CATBOOST_PARAMS = {
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1:average=Macro",
        "iterations": 3000,
        "learning_rate": 0.03,
        "depth": 8,
        "grow_policy": "SymmetricTree",
        "random_seed": SEED,
        "early_stopping_rounds": 150,
        "verbose": 200,
    }

    BIAS_STEPS = [1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002]

print(CFG.OUTPUT_DIR)

BASE_NUM_COLS = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]

BASE_CAT_COLS = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Mulching_Used",
    "Region",
]

ALL_BASE = BASE_NUM_COLS + BASE_CAT_COLS
PHYSICAL_COLS = ["ET_proxy", "water_deficit", "soil_quality"]
ORIG_PRIOR_COLS = [f"TE_ORIG_{c}" for c in ALL_BASE]
PAIRWISE_RAW_COLS = []
PAIRWISE_TE_COLS = []

experiments/exp_20260407_021_cat_no_pairwise_bias/artifacts


## utility

In [2]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch is not None:
        try:
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)
        except Exception:
            pass


def maybe_set_catboost_device(params: dict) -> dict:
    params = params.copy()
    use_gpu = False
    if torch is not None:
        try:
            use_gpu = torch.cuda.is_available()
        except Exception:
            use_gpu = False

    if use_gpu:
        params["task_type"] = "GPU"
        params["devices"] = "0"
    else:
        params["task_type"] = "CPU"
        params.pop("devices", None)
    return params


def ensure_output_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)

## Load data

In [3]:
def infer_original_path() -> str:
    if os.path.exists(CFG.ORIGINAL_PATH):
        return CFG.ORIGINAL_PATH

    candidates = [
        "/kaggle/input/playground-series-s6e4/original.csv",
        "/kaggle/input/playground-series-s6e4/train_original.csv",
        "/kaggle/input/playground-series-s6e4/original_data.csv",
    ]
    for c in candidates:
        if os.path.exists(c):
            return c

    raise FileNotFoundError(
        "original.csv のパスが見つかりません。CFG.ORIGINAL_PATH を自分の環境に合わせて修正してください。"
    )


def load_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df = pd.read_csv(CFG.TRAIN_PATH)
    test_df = pd.read_csv(CFG.TEST_PATH)
    original_path = infer_original_path()
    original_df = pd.read_csv(original_path)

    expected_train_cols = [CFG.ID_COL] + ALL_BASE + [CFG.TARGET_COL]
    expected_test_cols = [CFG.ID_COL] + ALL_BASE
    expected_original_cols = ALL_BASE + [CFG.TARGET_COL]

    missing_train = [c for c in expected_train_cols if c not in train_df.columns]
    missing_test = [c for c in expected_test_cols if c not in test_df.columns]
    missing_orig = [c for c in expected_original_cols if c not in original_df.columns]

    if missing_train:
        raise ValueError(f"train で不足列: {missing_train}")
    if missing_test:
        raise ValueError(f"test で不足列: {missing_test}")
    if missing_orig:
        raise ValueError(f"original で不足列: {missing_orig}")

    return train_df, test_df, original_df


def cast_base_categoricals(train_df: pd.DataFrame,
                           test_df: pd.DataFrame,
                           original_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df = train_df.copy()
    test_df = test_df.copy()
    original_df = original_df.copy()

    for col in BASE_CAT_COLS + [CFG.TARGET_COL]:
        if col in train_df.columns:
            train_df[col] = train_df[col].astype(str)
        if col in test_df.columns:
            test_df[col] = test_df[col].astype(str)
        if col in original_df.columns:
            original_df[col] = original_df[col].astype(str)

    return train_df, test_df, original_df

## physical features

In [4]:
def add_physical_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["ET_proxy"] = (
        out["Temperature_C"]
        * (100.0 - out["Humidity"]) / 100.0
        * (out["Sunlight_Hours"] + 1e-6)
    )
    out["water_deficit"] = out["ET_proxy"] - out["Rainfall_mm"] - out["Previous_Irrigation_mm"]
    out["soil_quality"] = (
        out["Organic_Carbon"] / (out["Electrical_Conductivity"] + 1e-6)
    ) * (out["Soil_Moisture"] + 1e-6)
    return out

## original priors

In [5]:
def _safe_multiclass_target_mean(series: pd.Series) -> float:
    codes, _ = pd.factorize(series.astype(str))
    return float(np.mean(codes)) if len(codes) > 0 else 0.0


def build_original_prior_mapping_for_categorical(original_df: pd.DataFrame,
                                                 col: str,
                                                 target_col: str) -> tuple[pd.Series, float]:
    mapping = original_df.groupby(col)[target_col].apply(_safe_multiclass_target_mean)
    global_mean = _safe_multiclass_target_mean(original_df[target_col])
    return mapping, global_mean


def build_original_prior_mapping_for_numeric(original_df: pd.DataFrame,
                                             col: str,
                                             target_col: str,
                                             n_bins: int = 10) -> tuple[np.ndarray, pd.Series, float]:
    x = original_df[col].astype(float)
    try:
        _, bin_edges = pd.qcut(x, q=n_bins, duplicates="drop", retbins=True)
    except ValueError:
        quantiles = np.linspace(0, 1, min(n_bins, max(2, x.nunique())) + 1)
        bin_edges = np.unique(np.quantile(x, quantiles))

    bin_edges = np.asarray(bin_edges, dtype=float)
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf

    orig_binned = pd.cut(x, bins=bin_edges, include_lowest=True)
    tmp = pd.DataFrame({col: orig_binned, target_col: original_df[target_col]})
    mapping = tmp.groupby(col, observed=False)[target_col].apply(_safe_multiclass_target_mean)
    global_mean = _safe_multiclass_target_mean(original_df[target_col])
    return bin_edges, mapping, global_mean


def add_original_target_priors(train_df: pd.DataFrame,
                               valid_df: pd.DataFrame,
                               test_df: pd.DataFrame,
                               original_df: pd.DataFrame,
                               target_col: str,
                               base_cols: list[str],
                               n_bins: int = 10) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df = train_df.copy()
    valid_df = valid_df.copy()
    test_df = test_df.copy()

    for col in base_cols:
        feat_name = f"TE_ORIG_{col}"

        if col in BASE_CAT_COLS:
            mapping, global_mean = build_original_prior_mapping_for_categorical(
                original_df, col, target_col
            )
            train_df[feat_name] = train_df[col].map(mapping).fillna(global_mean).astype(float)
            valid_df[feat_name] = valid_df[col].map(mapping).fillna(global_mean).astype(float)
            test_df[feat_name] = test_df[col].map(mapping).fillna(global_mean).astype(float)

        else:
            bin_edges, mapping, global_mean = build_original_prior_mapping_for_numeric(
                original_df, col, target_col, n_bins=n_bins
            )

            train_bin = pd.cut(train_df[col].astype(float), bins=bin_edges, include_lowest=True)
            valid_bin = pd.cut(valid_df[col].astype(float), bins=bin_edges, include_lowest=True)
            test_bin = pd.cut(test_df[col].astype(float), bins=bin_edges, include_lowest=True)

            # ここが重要:
            # Categorical のまま map -> fillna すると、新しい float category を入れられず落ちる
            train_mapped = pd.Series(train_bin.astype(object), index=train_df.index).map(mapping)
            valid_mapped = pd.Series(valid_bin.astype(object), index=valid_df.index).map(mapping)
            test_mapped = pd.Series(test_bin.astype(object), index=test_df.index).map(mapping)

            train_df[feat_name] = train_mapped.fillna(global_mean).astype(float)
            valid_df[feat_name] = valid_mapped.fillna(global_mean).astype(float)
            test_df[feat_name] = test_mapped.fillna(global_mean).astype(float)

    return train_df, valid_df, test_df

## make feature frame

In [6]:
def build_feature_frame(base_df: pd.DataFrame,
                        original_df: pd.DataFrame) -> pd.DataFrame:
    df = base_df.copy()

    if CFG.USE_PHYSICAL_FEATURES:
        df = add_physical_features(df)

    if CFG.USE_ORIG_PRIORS:
        no_target_df = df.drop(columns=[CFG.TARGET_COL], errors="ignore").copy()
        target_holder = df[[CFG.TARGET_COL]].copy() if CFG.TARGET_COL in df.columns else None
        train_side = df.copy() if CFG.TARGET_COL in df.columns else no_target_df.copy()

        if CFG.TARGET_COL in train_side.columns:
            train_side, _, _ = add_original_target_priors(
                train_side.copy(),
                train_side.copy(),
                no_target_df.copy(),
                original_df,
                CFG.TARGET_COL,
                ALL_BASE,
                n_bins=CFG.N_BINS_ORIG_PRIOR,
            )
            df = train_side
        else:
            dummy = no_target_df.copy()
            dummy[CFG.TARGET_COL] = original_df[CFG.TARGET_COL].iloc[0]
            _, _, test_aug = add_original_target_priors(
                dummy.copy(),
                dummy.copy(),
                no_target_df.copy(),
                original_df,
                CFG.TARGET_COL,
                ALL_BASE,
                n_bins=CFG.N_BINS_ORIG_PRIOR,
            )
            df = pd.concat([no_target_df.reset_index(drop=True), test_aug[ORIG_PRIOR_COLS].reset_index(drop=True)], axis=1)
            if target_holder is not None:
                df = pd.concat([target_holder.reset_index(drop=True), df.reset_index(drop=True)], axis=1)

    return df


def get_feature_cols() -> list[str]:
    feature_cols = []
    feature_cols += ALL_BASE

    if CFG.USE_PHYSICAL_FEATURES:
        feature_cols += PHYSICAL_COLS

    if CFG.USE_ORIG_PRIORS:
        feature_cols += ORIG_PRIOR_COLS

    assert len(PAIRWISE_RAW_COLS) == 0
    assert len(PAIRWISE_TE_COLS) == 0

    feature_cols = list(dict.fromkeys(feature_cols))
    return feature_cols


def get_cat_feature_cols(feature_cols: list[str]) -> list[str]:
    return [c for c in BASE_CAT_COLS if c in feature_cols]

## target encode/decode & bias tuning

In [7]:
def fit_label_encoder(y: pd.Series) -> tuple[dict[str, int], dict[int, str]]:
    classes = sorted(pd.Series(y).astype(str).unique().tolist())
    label_to_idx = {label: idx for idx, label in enumerate(classes)}
    idx_to_label = {idx: label for label, idx in label_to_idx.items()}
    return label_to_idx, idx_to_label


def encode_target(y: pd.Series, label_to_idx: dict[str, int]) -> np.ndarray:
    return y.astype(str).map(label_to_idx).values


def decode_pred_indices(pred_idx: np.ndarray, idx_to_label: dict[int, str]) -> list[str]:
    return [idx_to_label[int(i)] for i in pred_idx]


def multiclass_balanced_accuracy_from_proba(y_true_idx: np.ndarray, proba: np.ndarray) -> float:
    pred_idx = np.argmax(proba, axis=1)
    return float(balanced_accuracy_score(y_true_idx, pred_idx))


def apply_class_bias_logit(proba: np.ndarray, bias: np.ndarray) -> np.ndarray:
    eps = 1e-15
    logit = np.log(np.clip(proba, eps, 1.0 - eps))
    logit = logit + bias[None, :]
    x = np.exp(logit - logit.max(axis=1, keepdims=True))
    return x / x.sum(axis=1, keepdims=True)


def search_best_bias_multiclass(oof_proba: np.ndarray,
                                y_true_idx: np.ndarray,
                                steps: list[float]) -> tuple[np.ndarray, float]:
    n_classes = oof_proba.shape[1]
    best_bias = np.zeros(n_classes, dtype=float)
    best_score = multiclass_balanced_accuracy_from_proba(y_true_idx, oof_proba)

    current_best = best_bias.copy()
    for step in steps:
        improved = True
        while improved:
            improved = False
            candidates = [-step, 0.0, step]
            for delta in product(candidates, repeat=n_classes):
                trial = current_best + np.array(delta, dtype=float)
                tuned = apply_class_bias_logit(oof_proba, trial)
                score = multiclass_balanced_accuracy_from_proba(y_true_idx, tuned)
                if score > best_score:
                    best_score = score
                    best_bias = trial.copy()
                    current_best = trial.copy()
                    improved = True
    return best_bias, float(best_score)

## importance & submission

In [8]:
def get_feature_importance_frame(model: CatBoostClassifier,
                                 feature_cols: list[str]) -> pd.DataFrame:
    try:
        importances = model.get_feature_importance()
        fi = pd.DataFrame({
            "feature": feature_cols,
            "importance": importances,
        }).sort_values("importance", ascending=False).reset_index(drop=True)
    except Exception:
        fi = pd.DataFrame({"feature": feature_cols, "importance": np.nan})
    return fi


def build_submission(test_ids: pd.Series,
                     test_proba: np.ndarray,
                     idx_to_label: dict[int, str]) -> pd.DataFrame:
    pred_idx = np.argmax(test_proba, axis=1)
    pred_label = decode_pred_indices(pred_idx, idx_to_label)
    sub = pd.DataFrame({CFG.ID_COL: test_ids.values, CFG.TARGET_COL: pred_label})
    return sub

## 初期化とデータ読み込み

In [9]:
seed_everything(CFG.SEED)
ensure_output_dir(CFG.OUTPUT_DIR)
catboost_params = maybe_set_catboost_device(CFG.CATBOOST_PARAMS)

train_df, test_df, original_df = load_data()
train_df, test_df, original_df = cast_base_categoricals(train_df, test_df, original_df)

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
print("original shape:", original_df.shape)
print("catboost params:", catboost_params)

train shape: (630000, 21)
test shape: (270000, 20)
original shape: (10000, 20)
catboost params: {'loss_function': 'MultiClass', 'eval_metric': 'TotalF1:average=Macro', 'iterations': 3000, 'learning_rate': 0.03, 'depth': 8, 'grow_policy': 'SymmetricTree', 'random_seed': 42, 'early_stopping_rounds': 150, 'verbose': 200, 'task_type': 'GPU', 'devices': '0'}


## feature 準備

In [10]:
feature_cols = get_feature_cols()
cat_feature_cols = get_cat_feature_cols(feature_cols)
print("n_features =", len(feature_cols))
print("cat_feature_cols =", cat_feature_cols)

label_to_idx, idx_to_label = fit_label_encoder(train_df[CFG.TARGET_COL])
y_all = encode_target(train_df[CFG.TARGET_COL], label_to_idx)
original_y_all = encode_target(original_df[CFG.TARGET_COL], label_to_idx)
classes_ = [idx_to_label[i] for i in range(len(idx_to_label))]
print("classes =", classes_)

n_features = 41
cat_feature_cols = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
classes = ['High', 'Low', 'Medium']


## augmented frame 作成

In [11]:
original_aug = build_feature_frame(original_df.copy(), original_df)
train_aug_all = build_feature_frame(train_df.copy(), original_df)
test_aug_all = build_feature_frame(test_df.copy(), original_df)

print(original_aug[ORIG_PRIOR_COLS].dtypes.value_counts())
print(original_aug[ORIG_PRIOR_COLS].head())

assert all(col in original_aug.columns for col in feature_cols)
assert all(col in train_aug_all.columns for col in feature_cols)
assert all(col in test_aug_all.columns for col in feature_cols)

float64    19
Name: count, dtype: int64
   TE_ORIG_Soil_pH  TE_ORIG_Soil_Moisture  TE_ORIG_Organic_Carbon  TE_ORIG_Electrical_Conductivity  TE_ORIG_Temperature_C  TE_ORIG_Humidity  TE_ORIG_Rainfall_mm  TE_ORIG_Sunlight_Hours  \
0         0.442523               0.293293                0.462128                         0.478873               0.341000          0.455000                0.377                0.452642   
1         0.622419               0.752248                0.462128                         0.650000               0.567433          0.455000                0.652                0.655936   
2         0.453546               0.291417                0.488911                         0.478873               0.629259          0.421158                0.364                0.466135   
3         0.442523               0.606394                1.263527                         0.650000               0.567433          0.655000                0.652                0.650301   
4         0.453546  

## CV 学習本体

In [12]:
skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
n_classes = len(idx_to_label)

oof_proba = np.zeros((len(train_df), n_classes), dtype=float)
test_proba_folds = []
feature_importance_folds = []
fold_scores = []
models_best_iterations = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y_all), start=1):
    print("=" * 80)
    print(f"Fold {fold}/{CFG.N_FOLDS}")
    print("=" * 80)

    tr = train_aug_all.iloc[tr_idx].copy().reset_index(drop=True)
    va = train_aug_all.iloc[va_idx].copy().reset_index(drop=True)
    te = test_aug_all.copy().reset_index(drop=True)

    assert CFG.USE_PAIRWISE is False
    assert CFG.USE_PAIRWISE_TE is False
    assert len(PAIRWISE_RAW_COLS) == 0
    assert len(PAIRWISE_TE_COLS) == 0

    if CFG.USE_ORIGINAL_WEIGHTED_MERGE:
        X_tr = pd.concat([
            tr[feature_cols].copy(),
            original_aug[feature_cols].copy(),
        ], axis=0, ignore_index=True)
        y_tr = np.concatenate([
            encode_target(tr[CFG.TARGET_COL], label_to_idx),
            original_y_all,
        ])
        sample_weight = np.concatenate([
            np.ones(len(tr), dtype=float),
            np.full(len(original_aug), CFG.ORIG_ROW_WEIGHT, dtype=float),
        ])
    else:
        X_tr = tr[feature_cols].copy()
        y_tr = encode_target(tr[CFG.TARGET_COL], label_to_idx)
        sample_weight = np.ones(len(X_tr), dtype=float)

    X_va = va[feature_cols].copy()
    y_va = encode_target(va[CFG.TARGET_COL], label_to_idx)
    X_te = te[feature_cols].copy()

    cat_feature_indices = [X_tr.columns.get_loc(c) for c in cat_feature_cols]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_feature_indices, weight=sample_weight)
    valid_pool = Pool(X_va, y_va, cat_features=cat_feature_indices)
    test_pool = Pool(X_te, cat_features=cat_feature_indices)

    model = CatBoostClassifier(**catboost_params)
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    va_proba = model.predict_proba(valid_pool)
    te_proba = model.predict_proba(test_pool)

    oof_proba[va_idx] = va_proba
    test_proba_folds.append(te_proba)

    fold_score = multiclass_balanced_accuracy_from_proba(y_va, va_proba)
    fold_scores.append(fold_score)
    print(f"fold {fold} balanced_accuracy = {fold_score:.6f}")

    best_iter = getattr(model, "best_iteration_", None)
    models_best_iterations.append(best_iter)

    fi = get_feature_importance_frame(model, feature_cols)
    fi["fold"] = fold
    feature_importance_folds.append(fi)

Fold 1/5
0:	learn: 0.9658007	test: 0.9659570	best: 0.9659570 (0)	total: 179ms	remaining: 8m 58s
200:	learn: 0.9680516	test: 0.9679241	best: 0.9679824 (185)	total: 5.93s	remaining: 1m 22s
400:	learn: 0.9690129	test: 0.9680259	best: 0.9682178 (267)	total: 10.9s	remaining: 1m 10s
bestTest = 0.9682177922
bestIteration = 267
Shrink model to first 268 iterations.
fold 1 balanced_accuracy = 0.959114
Fold 2/5
0:	learn: 0.9663940	test: 0.9666705	best: 0.9666705 (0)	total: 49.9ms	remaining: 2m 29s
200:	learn: 0.9690840	test: 0.9683746	best: 0.9685261 (175)	total: 5.46s	remaining: 1m 16s
bestTest = 0.968526068
bestIteration = 175
Shrink model to first 176 iterations.
fold 2 balanced_accuracy = 0.960482
Fold 3/5
0:	learn: 0.9660773	test: 0.9662732	best: 0.9662732 (0)	total: 51.7ms	remaining: 2m 35s
200:	learn: 0.9692300	test: 0.9680862	best: 0.9680862 (200)	total: 5.5s	remaining: 1m 16s
400:	learn: 0.9701784	test: 0.9681290	best: 0.9681518 (371)	total: 10.4s	remaining: 1m 7s
600:	learn: 0.9708038	

## raw CV

In [13]:
raw_cv = multiclass_balanced_accuracy_from_proba(y_all, oof_proba)
print("raw_cv =", raw_cv)

test_proba = np.mean(test_proba_folds, axis=0)

raw_cv = 0.9596586272939867


## bias tuning

In [14]:
best_bias = np.zeros(n_classes, dtype=float)
biased_cv = None
oof_proba_biased = None
test_proba_biased = None

if CFG.USE_BIAS_TUNING:
    best_bias, biased_cv = search_best_bias_multiclass(oof_proba, y_all, CFG.BIAS_STEPS)
    oof_proba_biased = apply_class_bias_logit(oof_proba, best_bias)
    test_proba_biased = apply_class_bias_logit(test_proba, best_bias)
    print("best_bias =", best_bias)
    print("biased_cv =", biased_cv)

best_bias = [ 0.26 -2.79 -2.53]
biased_cv = 0.9659918395200521


## importance 集計

In [15]:
feature_importance_df = pd.concat(feature_importance_folds, axis=0, ignore_index=True)
feature_importance_mean = (
    feature_importance_df.groupby("feature", as_index=False)["importance"]
    .mean()
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

feature_columns_df = pd.DataFrame({"feature": feature_cols})

## submission 作成

In [16]:
submission_raw = build_submission(test_df[CFG.ID_COL], test_proba, idx_to_label)
submission_raw.to_csv(CFG.OUTPUT_DIR / "submission_cat_no_pairwise_bias_raw.csv", index=False)

if test_proba_biased is not None:
    submission_biased = build_submission(test_df[CFG.ID_COL], test_proba_biased, idx_to_label)
    submission_biased.to_csv(CFG.OUTPUT_DIR / "submission_cat_no_pairwise_bias.csv", index=False)

## 保存

In [17]:
np.save(CFG.OUTPUT_DIR / "oof_cat_no_pairwise_bias_proba.npy", oof_proba)
np.save(CFG.OUTPUT_DIR / "pred_cat_no_pairwise_bias_proba.npy", test_proba)

if oof_proba_biased is not None:
    np.save(CFG.OUTPUT_DIR / "oof_cat_no_pairwise_bias_proba_biased.npy", oof_proba_biased)
    np.save(CFG.OUTPUT_DIR / "pred_cat_no_pairwise_bias_proba_biased.npy", test_proba_biased)
    np.save(CFG.OUTPUT_DIR / "best_class_bias.npy", best_bias)

feature_columns_df.to_csv(CFG.OUTPUT_DIR / "feature_columns.csv", index=False)
feature_importance_df.to_csv(CFG.OUTPUT_DIR / "feature_importance_by_fold.csv", index=False)
feature_importance_mean.to_csv(CFG.OUTPUT_DIR / "feature_importance.csv", index=False)

cv_result = {
    "exp_id": CFG.EXP_ID,
    "raw_cv": float(raw_cv),
    "biased_cv": None if biased_cv is None else float(biased_cv),
    "fold_scores": [float(x) for x in fold_scores],
    "best_iterations": [None if x is None else int(x) for x in models_best_iterations],
    "n_features": int(len(feature_cols)),
    "feature_cols": feature_cols,
    "cat_feature_cols": cat_feature_cols,
    "use_physical_features": CFG.USE_PHYSICAL_FEATURES,
    "use_orig_priors": CFG.USE_ORIG_PRIORS,
    "use_original_weighted_merge": CFG.USE_ORIGINAL_WEIGHTED_MERGE,
    "orig_row_weight": float(CFG.ORIG_ROW_WEIGHT),
    "use_pairwise": CFG.USE_PAIRWISE,
    "use_pairwise_te": CFG.USE_PAIRWISE_TE,
    "seed": int(CFG.SEED),
    "n_folds": int(CFG.N_FOLDS),
    "catboost_params": catboost_params,
    "bias_steps": CFG.BIAS_STEPS,
    "classes": classes_,
}
with open(CFG.OUTPUT_DIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(cv_result, f, ensure_ascii=False, indent=2)

print("saved to:", CFG.OUTPUT_DIR)
print(feature_importance_mean.head(30))

saved to: experiments/exp_20260407_021_cat_no_pairwise_bias/artifacts
                            feature  importance
0         TE_ORIG_Crop_Growth_Stage   34.021858
1                     Soil_Moisture   25.431571
2                     Temperature_C   10.727580
3                    Wind_Speed_kmh   10.447205
4                     Mulching_Used   10.260129
5                 Crop_Growth_Stage    1.870718
6                       Rainfall_mm    1.423136
7             TE_ORIG_Mulching_Used    1.379346
8               TE_ORIG_Rainfall_mm    0.485623
9                     water_deficit    0.467851
10           Previous_Irrigation_mm    0.418765
11                         Humidity    0.399965
12            TE_ORIG_Soil_Moisture    0.317534
13                 TE_ORIG_Humidity    0.261139
14                          Soil_pH    0.217411
15   TE_ORIG_Previous_Irrigation_mm    0.186030
16          Electrical_Conductivity    0.170290
17                         ET_proxy    0.168126
18                

## 実行後チェック

In [18]:
print("raw_cv =", raw_cv)
print("biased_cv =", biased_cv)

print("pairwise cols in feature list:",
      [c for c in feature_cols if "pair" in c.lower() or "pairwise" in c.lower()])

print(feature_columns_df.head())
print(feature_columns_df.tail())

raw_cv = 0.9596586272939867
biased_cv = 0.9659918395200521
pairwise cols in feature list: []
                   feature
0                  Soil_pH
1            Soil_Moisture
2           Organic_Carbon
3  Electrical_Conductivity
4            Temperature_C
                    feature
36           TE_ORIG_Season
37  TE_ORIG_Irrigation_Type
38     TE_ORIG_Water_Source
39    TE_ORIG_Mulching_Used
40           TE_ORIG_Region
